In [ ]:
!pip -q install minerva-ml

In [2]:
from pathlib import Path
from PIL import Image
from typing import Any
import cv2
import random
import numpy as np
from minerva.data.readers.png_reader import PNGReader
from minerva.data.readers.patched_array_reader import NumpyArrayReader
from minerva.data.data_modules.base import MinervaDataModule

from minerva.transforms.transform import _Transform, Identity, Crop, Solarize, PadCrop, Padding, Flip, GrayScale, Solarize
from minerva.transforms.random_transform import _RandomSyncedTransform, RandomFlip, RandomGrayScale, RandomSolarize, RandomRotation
from minerva.data.datasets.base import SimpleDataset

from minerva.models.ssl.byol import BYOL
from torchvision import transforms
import lightning as L
import torch
from torchmetrics import JaccardIndex
import matplotlib.pyplot as plt
import torch.nn as nn
from lightning.pytorch.callbacks import Callback, ModelCheckpoint
from torchvision.models.resnet import resnet50

import copy

from google.colab import drive
import zipfile
import os

In [ ]:
drive.mount('/content/drive')

In [ ]:
DIR = '/content/drive/My Drive/'

In [4]:
class VerticalLineMask(_Transform):
    def __init__(self, min_lines: int = 3, max_lines: int = 5, line_width: int = 16, seed: int = None):
        """
        Creates a mask with random vertical lines of fixed width.

        Parameters
        ----------
        min_lines : int
            Minimum number of vertical lines to draw.
        max_lines : int
            Maximum number of vertical lines to draw.
        line_width : int
            Width of each vertical line in pixels.
        seed : int, optional
            Seed for reproducibility.
        """
        assert line_width > 0, "Line width must be positive."
        self.min_lines = min_lines
        self.max_lines = max_lines
        self.line_width = line_width
        self.rng = np.random.default_rng(seed)

    def __call__(self, x: np.ndarray) -> np.ndarray:
        if x.ndim == 2:
            h, w = x.shape
            c = 1
            format = 'HW'
        elif x.ndim == 3:
            if x.shape[0] == 1 or x.shape[0] == 3:
                c, h, w = x.shape
                format = 'CHW'
            else:
                h, w, c = x.shape
                format = 'HWC'
        else:
            raise ValueError(f"Unsupported input shape: {x.shape}")

        max_start = w - self.line_width
        if max_start <= 0:
            raise ValueError(f"Image width {w} too small for line width {self.line_width}")

        max_possible_lines = max_start // self.line_width
        num_lines = min(self.rng.integers(self.min_lines, self.max_lines + 1), max_possible_lines)

        possible_starts = np.arange(0, max_start + 1, self.line_width)
        start_positions = self.rng.choice(possible_starts, size=num_lines, replace=False)

        x_out = x.copy()

        if x.dtype == np.uint8:
            noise_max = 255
        else:
            noise_max = 1.0

        for start in start_positions:
            end = start + self.line_width
            if format == 'HW':
                x_out[:, start:end] = self.rng.random((h, self.line_width)) * 0
            elif format == 'HWC':
                x_out[:, start:end, :] = self.rng.random((h, self.line_width, c)) * 0
            elif format == 'CHW':
                x_out[:, :, start:end] = self.rng.random((c, h, self.line_width)) * 0

        return x_out

In [5]:
class HorizontalLineMask(_Transform):
    def __init__(self, min_lines: int = 3, max_lines: int = 5, line_width: int = 16, seed: int = None):
        """
        Creates a mask with random horizontal lines of fixed width.

        Parameters
        ----------
        min_lines : int
            Minimum number of horizontal lines to draw.
        max_lines : int
            Maximum number of horizontal lines to draw.
        line_width : int
            Height of each horizontal line in pixels.
        seed : int, optional
            Seed for reproducibility.
        """
        assert line_width > 0,
        self.min_lines = min_lines
        self.max_lines = max_lines
        self.line_width = line_width
        self.rng = np.random.default_rng(seed)

    def __call__(self, x: np.ndarray) -> np.ndarray:
        if x.ndim == 2:
            h, w = x.shape
            c = 1
            format = 'HW'
        elif x.ndim == 3:
            if x.shape[0] == 1 or x.shape[0] == 3:
                c, h, w = x.shape
                format = 'CHW'
            else:
                h, w, c = x.shape
                format = 'HWC'
        else:
            raise ValueError(f"Unsupported input shape: {x.shape}")

        max_start = h - self.line_width
        if max_start <= 0:
            raise ValueError(f"Image height {h} too small for line width {self.line_width}")

        max_possible_lines = max_start // self.line_width
        num_lines = min(self.rng.integers(self.min_lines, self.max_lines + 1), max_possible_lines)

        possible_starts = np.arange(0, max_start + 1, self.line_width)
        start_positions = self.rng.choice(possible_starts, size=num_lines, replace=False)

        x_out = x.copy()

        if x.dtype == np.uint8:
            noise_max = 255
        else:
            noise_max = 1.0

        for start in start_positions:
            end = start + self.line_width
            if format == 'HW':
                x_out[start:end, :] = self.rng.random((self.line_width, w)) * 0
            elif format == 'HWC':
                x_out[start:end, :, :] = self.rng.random((self.line_width, w, c)) * 0
            elif format == 'CHW':
                x_out[:, start:end, :] = self.rng.random((c, self.line_width, w)) * 0

        return x_out

In [6]:
class Identity_2(_Transform):

    def __call__(self, x: np.ndarray) -> np.ndarray:
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        transform = transforms.Compose([transforms.ToTensor(), normalize])

        x1, x2 = x
        x1 = transform(x1)
        x2 = transform(x2)
        return (x1,x2)

    def __str__(self) -> str:
        return "Identity()"

In [7]:
class AdditiveGaussianNoise(_Transform):
    def __init__(self, mean: float = 0.0, std: float = 0.0):
        """
        mean: noise mean (0.0 is neutral)
        std: standard deviation in pixel intensity (0–255 scale)
        p: probability to apply the transform
        """
        self.mean = mean
        self.std = std

    def __call__(self, x: np.ndarray) -> np.ndarray:

        noise = np.random.normal(self.mean, self.std, size=x.shape).astype(np.float32)
        noisy = x.astype(np.float32) + noise
        return np.clip(noisy, 0, 255).astype(np.uint8)

In [8]:
class RandomTransform(_Transform):
    def __init__(self, transform_1: _Transform, transform_2: _Transform, transform_3: _Transform, transform_4: _Transform, rng: float = 0.0):
        self.transform_1 = transform_1
        self.transform_2 = transform_2
        self.transform_3 = transform_3
        self.transform_4 = transform_4
        self.rng = rng

    def __call__(self, x: Any) -> Any:
        if random.random() < self.rng:
            x1 = self.transform_1(x)
            x2 = self.transform_2(x)
        else:
            x1 = self.transform_3(x)
            x2 = self.transform_4(x)
        return (x1,x2)

In [ ]:
class RandomTransform_2(_Transform):
    def __init__(self, transform_1: _Transform, transform_2: _Transform, transform_3: _Transform, transform_4: _Transform,
                 transform_5: _Transform, transform_6: _Transform, rng = []):
        self.transform_1 = transform_1
        self.transform_2 = transform_2
        self.transform_3 = transform_3
        self.transform_4 = transform_4
        self.transform_5 = transform_5
        self.transform_6 = transform_6
        self.rng = rng

    def __call__(self, x: Any) -> Any:
        probability = random.random()
        if probability < self.rng[0]:
            x1 = self.transform_1(x)
            x2 = self.transform_1(x)
        if probability < self.rng[1]:
            x1 = self.transform_2(x1)
            x2 = self.transform_2(x2)
        if probability < self.rng[2]:
            x1 = self.transform_3(x1)
            x2 = self.transform_3(x2)
        if probability < self.rng[3]:
            x1 = self.transform_4(x1)
            x2 = self.transform_4(x2)
        if probability < self.rng[4]:
            x1 = self.transform_5(x1)
        if probability < self.rng[5]:
            x2 = self.transform_5(x2)
        if probability < self.rng[6]:
            x2 = self.transform_6(x2)
        return (x1,x2)

In [ ]:
class ColorJitter(_Transform):
    def __init__(
        self,
        brightness: float = 0.0,
        contrast: float = 0.0,
        saturation: float = 0.0,
        hue: float = 0.0,
    ):
        """
        Applies fixed adjustments to brightness, contrast, saturation, and hue to an input image.

        Parameters
        ----------
        brightness : float, optional
            Fixed factor for brightness adjustment. A value of 1.0 means no change. Defaults to 1.0.
        contrast : float, optional
            Fixed factor for contrast adjustment. A value of 1.0 means no change. Defaults to 1.0.
        saturation : float, optional
            Fixed factor for saturation adjustment. A value of 1.0 means no change. Defaults to 1.0.
        hue : float, optional
            Fixed degree shift for hue adjustment, in the range [-180, 180]. Defaults to 0.0.

        Returns
        -------
        np.ndarray
            The transformed image with fixed brightness, contrast, saturation, and hue adjustments applied.
        """
        self.brightness = random.uniform(1.0, 2.0)
        self.contrast = random.uniform(1.0, 2.0)
        self.saturation = random.uniform(1.0, 2.0)
        self.hue = random.uniform(1.0, 2.0)

    def __call__(self, image: np.ndarray) -> np.ndarray:
        rng = random.random()
        # Convert to HSV for hue/saturation adjustment
        image = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)

        # Brightness adjustment
        if rng <= 0.4:
            brightness = 10
        else:
            brightness = random.uniform(1.0, 2.0)
        image[..., 2] = np.clip(image[..., 2] * brightness, 0, 255)

        # Saturation adjustment
        if rng <= 0.4:
            contrast = 10
        else:
            contrast = random.uniform(1.0, 2.0)
        image[..., 1] = np.clip(image[..., 1] * contrast, 0, 255)

        # Contrast adjustment
        if rng <= 0.2:
            saturation = 10
        else:
            saturation = random.uniform(1.0, 2.0)
        mean = image[..., 2].mean()
        image[..., 2] = np.clip((image[..., 2] - mean) * saturation + mean, 0, 255)

        # Hue adjustment
        if rng <= 0.1:
            hue = 10
        else:
            hue = random.uniform(1.0, 3.0)
        image[..., 0] = (image[..., 0] + hue) % 180

        return cv2.cvtColor(image.astype(np.uint8), cv2.COLOR_HSV2RGB)

    def __str__(self) -> str:
        return f"ColorJitter(brightness={self.brightness}, contrast={self.contrast}, saturation={self.saturation}, hue={self.hue})"

In [ ]:
class RandomCrop(_Transform):

    def __init__(self, resize_size: tuple):

        self.resize_size = resize_size

    def __call__(self, img: np.ndarray) -> np.ndarray:
        """
        Randomly crop and resize an RGB image using cv2.

        Args:
            img (np.ndarray): Input image (H, W, 3).
            crop_size (tuple): (crop_width, crop_height) -> size of the crop.
            resize_size (tuple): (resize_width, resize_height) -> size to resize after crop.

        Returns:
            np.ndarray: Cropped and resized image (resize_height, resize_width, 3).
        """
        h, w, _ = img.shape
        crop_perc = random.uniform(0.08, 1.0)
        crop_w = round(img.shape[0]*crop_perc)
        crop_h = round(img.shape[1]*crop_perc)

        # Ensure crop fits inside original image
        crop_w = min(crop_w, w)
        crop_h = min(crop_h, h)

        # Random top-left corner
        start_x = random.randint(0, w - crop_w)
        start_y = random.randint(0, h - crop_h)

        # Crop
        cropped = img[start_y:start_y+crop_h, start_x:start_x+crop_w]

        # Resize with cv2 (INTER_AREA is good for shrinking, INTER_LINEAR for enlarging)
        resized = cv2.resize(cropped, self.resize_size, interpolation=cv2.INTER_LINEAR)

        return resized

# Reader

In [ ]:
# reader loads the dataset to be used in the model
# you need to write the path of the dataset in the function PNGReader(path=Path('path to dataset'))

In [9]:
!unzip -q '/content/drive/My Drive/spectogram_rgb.zip' -d'/content/'

In [10]:
train_data_reader = PNGReader(path=Path('/content/spectogram_rgb'))

In [ ]:
!unzip -q '/content/drive/My Drive/dataset_no_panoradio_rgb.zip' -d'/content/'

In [ ]:
train_data_reader_0 = PNGReader(path=Path('/content/dataset_no_panoradio_rgb'))

In [ ]:
!unzip -q '/content/drive/My Drive/dataset_no_powder_rgb.zip' -d'/content/'

In [ ]:
train_data_reader_1 = PNGReader(path=Path('/content/dataset_no_powder_rgb'))

# dataset + reader

In [ ]:
dataset_0 = SimpleDataset(
        readers=[train_data_reader_0, train_data_reader_0],
        transforms=[VerticalLineMask(), AdditiveGaussianNoise(std=60)],
        return_single = False
  )

In [ ]:
dataset_1 = SimpleDataset(
        readers=[train_data_reader, train_data_reader],
        transforms=[VerticalLineMask(), AdditiveGaussianNoise(std=60)],
        return_single = False
  )

In [ ]:
dataset_2 = SimpleDataset(
        readers=[train_data_reader, train_data_reader],
        transforms=[HorizontalLineMask(), ColorJitter()],
        return_single = False
    )

In [ ]:
dataset_3 = SimpleDataset(
        readers=[train_data_reader, train_data_reader],
        transforms=[VerticalLineMask(), ColorJitter()],
        return_single = False
  )

In [ ]:
dataset_4 = SimpleDataset(
        readers=[train_data_reader, train_data_reader],
        transforms=[HorizontalLineMask(), AdditiveGaussianNoise(std=60)],
        return_single = False
    )

In [ ]:
dataset_5 = SimpleDataset(
        readers=[train_data_reader],
        transforms=[RandomTransform(VerticalLineMask(), ColorJitter(),
                                   HorizontalLineMask(), AdditiveGaussianNoise(std=60), 0.5)],
        return_single = True
    )

In [ ]:
dataset_6 = SimpleDataset(
        readers=[train_data_reader],
        transforms=[RandomTransform(VerticalLineMask(), AdditiveGaussianNoise(std=60),
                                   HorizontalLineMask(), ColorJitter()],
        return_single = True
    )

In [ ]:
dataset_7 = SimpleDataset(
        readers=[train_data_reader],
        transforms=[RandomTransform(VerticalLineMask(), ColorJitter(),
                                   HorizontalLineMask(), ColorJitter(), 0.5)],
        return_single = True
    )

In [11]:
dataset_8 = SimpleDataset(
        readers=[train_data_reader],
        transforms=[RandomTransform(VerticalLineMask(), AdditiveGaussianNoise(std=60),
                                   HorizontalLineMask(), AdditiveGaussianNoise(std=60), 0.5)],
        return_single = True
    )

In [ ]:
dataset_9 = SimpleDataset(
        readers=[train_data_reader],
        transforms=[RandomTransform_2(RandomCrop((256,256)), Flip(1), ColorJitter(), GrayScale(), AdditiveGaussianNoise(std=60), Solarize(),
                    rng = [1.0, 0.5, 0.8, 0.2, 1.0, 0.1, 0.2])],
        return_single = True
    )

# dataset config

In [ ]:
train_dataset_config_0 = dataset_0

In [ ]:
train_dataset_config_1 = dataset_1

In [ ]:
train_dataset_config_2 = dataset_2

In [ ]:
train_dataset_config_3 = dataset_3

In [ ]:
train_dataset_config_4 = dataset_4

In [ ]:
train_dataset_config_5 = dataset_5

In [ ]:
train_dataset_config_6 = dataset_6

In [12]:
train_dataset_config_7 = dataset_7

In [ ]:
train_dataset_config_8 = dataset_8

In [ ]:
train_dataset_config_9 = dataset_9

# dataset config + normalize


In [ ]:
train_dataset_0 = SimpleDataset(
        readers=[train_dataset_config_0],
        transforms=[Identity_2()],
        return_single = True
    )

In [ ]:
train_dataset_1 = SimpleDataset(
        readers=[train_dataset_config_1],
        transforms=[Identity_2()],
        return_single = True
    )

In [ ]:
train_dataset_2 = SimpleDataset(
        readers=[train_dataset_config_2],
        transforms=[Identity_2()],
        return_single = True
    )

In [ ]:
train_dataset_3 = SimpleDataset(
        readers=[train_dataset_config_3],
        transforms=[Identity_2()],
        return_single = True
    )

In [ ]:
train_dataset_4 = SimpleDataset(
        readers=[train_dataset_config_4],
        transforms=[Identity_2()],
        return_single = True
    )

In [ ]:
train_dataset_5 = SimpleDataset(
        readers=[train_dataset_config_5],
        transforms=[Identity_2()],
        return_single = True
    )

In [ ]:
train_dataset_6 = SimpleDataset(
        readers=[train_dataset_config_6],
        transforms=[Identity_2()],
        return_single = True
    )

In [13]:
train_dataset_7 = SimpleDataset(
        readers=[train_dataset_config_7],
        transforms=[Identity_2()],
        return_single = True
    )

In [ ]:
train_dataset_8 = SimpleDataset(
        readers=[train_dataset_config_8],
        transforms=[Identity_2()],
        return_single = True
    )

In [ ]:
train_dataset_9 = SimpleDataset(
        readers=[train_dataset_config_9],
        transforms=[Identity_2()],
        return_single = True
    )

# data module

In [ ]:
data_module_0 = MinervaDataModule(
        train_dataset=train_dataset_0,
        batch_size=128,
        num_workers=8,
        name="Spectogram Dataset"
    )

In [ ]:
data_module_1 = MinervaDataModule(
        train_dataset=train_dataset_1,
        batch_size=128,
        num_workers=8,
        name="Spectogram Dataset"
    )

In [ ]:
data_module_2 = MinervaDataModule(
        train_dataset=train_dataset_2,
        batch_size=128,
        num_workers=8,
        name="Spectogram Dataset"
    )

In [ ]:
data_module_3 = MinervaDataModule(
        train_dataset=train_dataset_3,
        batch_size=128,
        num_workers=8,
        name="Spectogram Dataset"
    )

In [ ]:
data_module_4 = MinervaDataModule(
        train_dataset=train_dataset_4,
        batch_size=128,
        num_workers=8,
        name="Spectogram Dataset"
    )

In [ ]:
data_module_5 = MinervaDataModule(
        train_dataset=train_dataset_5,
        batch_size=128,
        num_workers=8,
        name="Spectogram Dataset"
    )

In [ ]:
data_module_6 = MinervaDataModule(
        train_dataset=train_dataset_6,
        batch_size=128,
        num_workers=8,
        name="Spectogram Dataset"
    )

In [14]:
data_module_7 = MinervaDataModule(
        train_dataset=train_dataset_7,
        batch_size=128,
        num_workers=8,
        name="Spectogram Dataset"
    )

In [ ]:
data_module_8 = MinervaDataModule(
        train_dataset=train_dataset_8,
        batch_size=128,
        num_workers=8,
        name="Spectogram Dataset"
    )

In [ ]:
data_module_9 = MinervaDataModule(
        train_dataset=train_dataset_9,
        batch_size=128,
        num_workers=8,
        name="Spectogram Dataset"
    )

# modelos

In [ ]:
RN50model_none_dilatation = resnet50(weights=None,replace_stride_with_dilation=[False, False, True])

In [ ]:
model_0 = BYOL(
    backbone = torch.nn.Sequential(*list(copy.deepcopy(RN50model_none_dilatation).children())[:-1]),
    learning_rate=1e-4
)

In [ ]:
model_1 = BYOL(
    backbone = torch.nn.Sequential(*list(copy.deepcopy(RN50model_none_dilatation).children())[:-1]),
    learning_rate=1e-4
)

In [ ]:
model_2 = BYOL(
    backbone = torch.nn.Sequential(*list(copy.deepcopy(RN50model_none_dilatation).children())[:-1]),
    learning_rate=1e-4
)

In [ ]:
model_3 = BYOL(
    backbone = torch.nn.Sequential(*list(copy.deepcopy(RN50model_none_dilatation).children())[:-1]),
    learning_rate=1e-4
)

In [ ]:
model_4 = BYOL(
    backbone = torch.nn.Sequential(*list(copy.deepcopy(RN50model_none_dilatation).children())[:-1]),
    learning_rate=1e-4
)

In [ ]:
model_5 = BYOL(
    backbone = torch.nn.Sequential(*list(copy.deepcopy(RN50model_none_dilatation).children())[:-1]),
    learning_rate=1e-4
)

In [ ]:
model_6 = BYOL(
    backbone = torch.nn.Sequential(*list(copy.deepcopy(RN50model_none_dilatation).children())[:-1]),
    learning_rate=1e-4
)

In [16]:
model_7 = BYOL(
    backbone = torch.nn.Sequential(*list(copy.deepcopy(RN50model_none_dilatation).children())[:-1]),
    learning_rate=1e-4
)

In [ ]:
model_8 = BYOL(
    backbone = torch.nn.Sequential(*list(copy.deepcopy(RN50model_none_dilatation).children())[:-1]),
    learning_rate=1e-4
)

In [ ]:
model_9 = BYOL(
    backbone = torch.nn.Sequential(*list(copy.deepcopy(RN50model_none_dilatation).children())[:-1]),
    learning_rate=1e-4
)

# checkpoint model

In [ ]:
# dirpath = "path where the checkpoints will be saved"

In [ ]:
checkpoint_0 = ModelCheckpoint(
    every_n_epochs=5,
    save_top_k=-1,
    dirpath=DIR + "checkpoints_byol",
    filename="config_0_epoch{epoch:02d}"
  )

In [ ]:
checkpoint_1 = ModelCheckpoint(
    every_n_epochs=5,
    save_top_k=-1,
    dirpath=DIR + "checkpoints_byol",
    filename="config_1_epoch{epoch:02d}"
  )

In [ ]:
checkpoint_2 = ModelCheckpoint(
    every_n_epochs=5,
    save_top_k=-1,
    dirpath=DIR + "checkpoints_byol",
    filename="config_2_epoch{epoch:02d}"
  )

In [ ]:
checkpoint_3 = ModelCheckpoint(
    every_n_epochs=5,
    save_top_k=-1,
    dirpath=DIR + "checkpoints_byol",
    filename="config_3_epoch{epoch:02d}"
  )

In [ ]:
checkpoint_4 = ModelCheckpoint(
    every_n_epochs=5,
    save_top_k=-1,
    dirpath=DIR + "checkpoints_byol",
    filename="config_4_epoch{epoch:02d}"
  )

In [ ]:
checkpointe_5 = ModelCheckpoint(
    every_n_epochs=5,
    save_top_k=-1,
    dirpath=DIR + "checkpoints_byol",
    filename="config_5_epoch{epoch:02d}"
  )

In [ ]:
checkpoint_6 = ModelCheckpoint(
    every_n_epochs=5,
    save_top_k=-1,
    dirpath=DIR + "checkpoints_byol",
    filename="config_6_epoch{epoch:02d}"
  )

In [17]:
checkpoint_7 = ModelCheckpoint(
    every_n_epochs=5,
    save_top_k=-1,
    dirpath=DIR + "checkpoints_byol",
    filename="config_7_epoch{epoch:02d}"
  )

In [ ]:
checkpoint_8 = ModelCheckpoint(
    every_n_epochs=5,
    save_top_k=-1,
    dirpath=DIR + "checkpoints_byol",
    filename="config_8_epoch{epoch:02d}"
  )

In [ ]:
checkpoint_callback_9 = ModelCheckpoint(
    every_n_epochs=5,
    save_top_k=-1,
    dirpath=DIR + "checkpoints_byol",
    filename="config_9_epoch{epoch:02d}"
  )

# trainer

In [ ]:
trainer_0 = L.Trainer(
        max_epochs=30,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback_none_0]
    )

In [ ]:
trainer_1 = L.Trainer(
        max_epochs=30,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback_none_1]
    )

In [ ]:
trainer_2 = L.Trainer(
        max_epochs=50,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback_none_2]
    )

In [ ]:
trainer_3 = L.Trainer(
        max_epochs=50,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback_none_3]
    )

In [ ]:
trainer_4 = L.Trainer(
        max_epochs=50,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback_none_4]
    )

In [ ]:
trainer_5 = L.Trainer(
        max_epochs=30,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback_none_5]
    )

In [ ]:
trainer_6 = L.Trainer(
        max_epochs=30,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback_default_6]
    )

In [ ]:
trainer_7 = L.Trainer(
        max_epochs=30,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback_default_7]
    )

In [ ]:
trainer_8 = L.Trainer(
        max_epochs=30,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback_default_8]
    )

In [ ]:
trainer_9 = L.Trainer(
        max_epochs=30,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback_9]
    )

# training

In [ ]:
trainer_0.fit(model_0, data_module_0)

In [ ]:
trainer_1.fit(model_1, data_module_1)

In [ ]:
trainer_2.fit(model_2, data_module_2)

In [ ]:
trainer_3.fit(model_3, data_module_3)

In [ ]:
trainer_4.fit(model_4, data_module_4)

In [ ]:
trainer_5.fit(model_5, data_module_5)

In [ ]:
trainer_6.fit(model_6, data_module_6)

In [ ]:
trainer_7.fit(model_7, data_module_7)

In [ ]:
trainer_8.fit(model_8, data_module_8)

In [ ]:
trainer_9.fit(model_9, data_module_9)